# ML-10 — Content Action Playbook

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

*Read `skills/README.md`, then loaded `skills/writing-honest-claims/SKILL.md` and `skills/flyrank/flyrank-data/SKILL.md` as directed on this assignment's card. This playbook is built on the W04 baseline rule, not the W05 model — W06's honest audit found the baseline still wins on a non-circular evaluation, so this is the currently-defensible version, not the most complex one.*

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [2]:
import os

if not os.path.isdir("ml-internship"):
    !git clone https://github.com/m-husnain-dev/ml-internship.git

os.chdir("ml-internship/work/notebooks")
print("Current directory:", os.getcwd())

Cloning into 'ml-internship'...
remote: Enumerating objects: 197, done.
remote: Counting objects: 100% (197/197), done.
remote: Compressing objects: 100% (168/168), done.
remote: Total 197 (delta 97), reused 75 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (197/197), 1.91 MiB | 26.03 MiB/s, done.
Resolving deltas: 100% (97/97), done.
Current directory: /content/ml-internship/work/notebooks


In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
eligible = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
eligible["is_declining_label"] = (eligible["trend_direction"] == "down").astype(int)

# The W04 baseline rule, unchanged — per W06, it's still the honestly-validated best option.
declining_flag = (eligible["trend_direction"] == "down").astype(int)
visible_flag = (eligible["impressions_90d"] >= 500).astype(int)
stale_flag = ((eligible["days_since_last_update"] >= 30) & (eligible["days_since_last_update"] < 110)).astype(int)
low_ctr_flag = ((eligible["ctr"] < 0.5) & (eligible["avg_position"] > 0) & (eligible["avg_position"] <= 20)).astype(int)

eligible["score"] = declining_flag * visible_flag * eligible["impressions_90d"] * (1 + 0.5*stale_flag + 0.5*low_ctr_flag)

def reason_code(low_ctr, stale):
    if low_ctr:
        return "low_ctr_visible_page"
    elif stale:
        return "moderately_stale_page"
    else:
        return "declining_with_demand"

eligible["reason_code"] = [reason_code(l, s) for l, s in zip(low_ctr_flag, stale_flag)]

# Archetype -> action mapping.
ARCHETYPE_ACTION = {
    "low_ctr_visible_page": "rewrite_title_meta",
    "moderately_stale_page": "content_refresh_review",
    "declining_with_demand": "monitor_or_review",
}
eligible["action"] = eligible["reason_code"].map(ARCHETYPE_ACTION)
eligible.loc[eligible["score"] == 0, "action"] = "monitor"

ranked = eligible.sort_values("score", ascending=False).reset_index(drop=True)
print(f"Flagged for review: {(ranked['score'] > 0).sum()} out of {len(ranked)}")
print()
print(ranked.loc[ranked['score']>0, 'reason_code'].value_counts())
print()
ranked[["content_id","score","reason_code","action","impressions_90d","ctr","avg_position","days_since_last_update"]].head(10)

Flagged for review: 9961 out of 30000

reason_code
low_ctr_visible_page     6120
declining_with_demand    2094
moderately_stale_page    1747
Name: count, dtype: int64



,content_id,score,reason_code,action,impressions_90d,ctr,avg_position,days_since_last_update
0,content_5fe46e04994d,1035430.0,low_ctr_visible_page,rewrite_title_meta,517715,0.14,4.2,104
1,content_8c19996aa890,763878.0,low_ctr_visible_page,rewrite_title_meta,509252,0.15,2.5,20
2,content_4c36c775b818,694654.5,low_ctr_visible_page,rewrite_title_meta,463103,0.41,2.3,20
3,content_1a9e894be2e2,624270.0,low_ctr_visible_page,rewrite_title_meta,416180,0.23,4.0,22
4,content_cb112fce36be,619820.0,low_ctr_visible_page,rewrite_title_meta,309910,0.16,5.6,104
5,content_2c2606c5d176,521098.5,moderately_stale_page,content_refresh_review,347399,0.53,4.2,104
6,content_9532f197bbc8,463788.0,moderately_stale_page,content_refresh_review,309192,0.87,2.0,104
7,content_c8e9d6ab9013,417356.0,low_ctr_visible_page,rewrite_title_meta,208678,0.00,9.7,104
8,content_3d94572c3a35,381246.0,low_ctr_visible_page,rewrite_title_meta,190623,0.24,4.3,104
9,content_01908772c6db,375786.0,low_ctr_visible_page,rewrite_title_meta,187893,0.45,4.0,104


**Archetype → action mapping, in words a reviewer can trust:**
- `low_ctr_visible_page` → **rewrite_title_meta**: the page ranks well enough to be seen but isn't converting that visibility into clicks — a title/snippet problem, not a content-quality one.
- `moderately_stale_page` → **content_refresh_review**: hasn't been touched in 30–110 days; per W04's signal check this is a *mixed* signal (see the decay/refresh insight below), so it's flagged for review, not automatic action.
- `declining_with_demand` → **monitor_or_review**: declining and visible, but without a specific secondary problem identified — lowest-confidence category, worth a human look before any real time investment.

**The decay/refresh insight (from W04's own signal check, re-shown here):**

In [5]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
os.makedirs("../figures", exist_ok=True)
bins = [0, 30, 110, 100000]
bucket_labels = ["fresh (<30d)", "moderate (30-110d)", "stale (110d+)"]
eligible["staleness_bucket"] = pd.cut(eligible["days_since_last_update"], bins=bins, labels=bucket_labels, right=False)
staleness_table = eligible.groupby("staleness_bucket", observed=True).agg(
    n=("is_declining_label", "count"), declining_rate=("is_declining_label", "mean")
).round(3)
print(staleness_table)

fig, ax = plt.subplots(figsize=(6,4))
ax.bar(staleness_table.index.astype(str), staleness_table["declining_rate"], color=["#3B5BDB","#FF6B4A","#3B5BDB"])
ax.set_ylabel("Decline rate")
ax.set_title("Decline rate by staleness bucket (mixed signal, n shown)")
for i, (n, rate) in enumerate(zip(staleness_table["n"], staleness_table["declining_rate"])):
    ax.text(i, rate + 0.01, f"n={n}", ha="center", fontsize=9)
plt.tight_layout()
plt.savefig("../figures/staleness_decline_rate.png", dpi=150)
plt.show()
print("Figure saved to work/figures/staleness_decline_rate.png")

                        n  declining_rate
staleness_bucket                         
fresh (<30d)        20480           0.511
moderate (30-110d)   9290           0.613
stale (110d+)         230           0.426
Figure saved to work/figures/staleness_decline_rate.png


**Honest reading of this chart:** decline rate rises from fresh (0.511) to moderately stale (0.613) — the expected direction — but drops again in the stale bucket (0.426, n=230, a much smaller sample). Per the writing-honest-claims skill, a small bucket doesn't get to carry a headline claim, so the safe statement is: *staleness in the 30–110 day range is associated with a somewhat higher observed decline rate in this dataset; the pattern does not continue cleanly into the 110+ day range, and that tail is too small to trust on its own.* This is why staleness is used as a secondary priority boost in the rule, never the primary gate.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended use:** a content reviewer with limited time uses this ranked queue to decide which pages to look at first, out of thousands. The output is a prioritized worklist with a reason attached to each item — not a verdict, and not an automatic action.

**In safe claim-ladder language:** these pages *look worth reviewing first*, because they show declining search performance alongside real traffic at stake and, for some, a specific measurable problem (CTR or staleness). That is decision-support language, not a causal claim — nothing here says a refresh *will* fix a page, only that the pattern in this dataset makes it *worth a human's time to check*.

**Limits:**
- Built and validated on the 30,000-row anonymized **starter dataset only** (32 clients) — not the full warehouse. Numbers here should not be presented as portfolio-wide until re-validated at scale.
- This playbook is built on the **W04 baseline rule**, not the W05/W06 model. W06's honest audit (client-grouped split, non-circular `true_priority` target) found the baseline still outperforms the model attempted so far — so shipping the model here would be shipping the *worse* option to look more sophisticated. That's exactly the "reward complexity, not results" mistake the track has been warning against all along.
- This is a **cross-sectional, observational** dataset (one snapshot, no intervention run) — per the writing-honest-claims skill, that means it can never by itself support "doing X will produce Y." Only a real refresh-vs-no-refresh experiment could support a causal claim.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**What a human must check before acting on any flagged row:**
- Sanity-check the reason code against the real numbers in that row — W04's own top-20 review found a genuine weak pick: a page with a healthy 0.87% CTR and a strong position 2.0 still got flagged `moderately_stale_page` purely on time-since-update. A reviewer should discard picks like that on sight, not action them.
- Confirm `avg_position` isn't beyond the tier the reason code assumes — W04 also found a page at position 26.2 (page 3+) flagged as if a title rewrite would help, when a position that deep may need a ranking push instead, a different fix entirely.
- Confirm the data is current — a page may have already been refreshed since this snapshot was taken; the queue doesn't know that.

**The no-go list — what must never be automated here:**
- **Never auto-publish, auto-rewrite, or auto-consolidate/redirect any page.** This system ranks candidates for a human to look at; it does not act on anything itself.
- **Never present this as proof a refresh will improve a page.** No experiment was run — only an observational comparison. That claim would violate the claim ladder outright.
- **Never treat the `stale (110d+)` bucket finding as strong evidence for anything** — n=230 is too small per the writing-honest-claims skill's own banned-phrasing rule against headline ratios from tiny buckets.
- **Never ship this queue to a client-facing report without a human sign-off first** — the known weak-pick pattern above means a share of any top-K list will be wrong by construction, and a client audience won't know to discount that the way an internal reviewer would.

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**Signals worth monitoring over time, if this were running on a rolling basis:**
- **Base rate drift:** if the overall decline rate (currently 54.2% of eligible pages) shifts meaningfully, the rule's assumptions about what counts as "normal" may no longer hold.
- **Reason code distribution drift:** a sudden shift in the mix of `low_ctr_visible_page` vs `moderately_stale_page` vs `declining_with_demand` could signal an underlying change (e.g. a search-result-page layout change affecting CTR broadly, not page-by-page).
- **CTR-by-position-tier drift:** the whole `low_ctr_visible_page` reason code depends on the tier-based CTR pattern confirmed in W01/W04 (0.355% → 0.055% across tiers). If a SERP feature change shifts that baseline pattern, the 0.5% cutoff used here would need re-deriving, not just re-applying.

**Retrain / re-validate trigger:** if a future model is trained *directly* on the `true_priority` target (not the proxy `is_declining_label` W05 used) and beats this baseline under a client-grouped split — the exact next step W06's audit pointed to — that would be the trigger to promote a model over this rule. Until that comparison exists and wins honestly, the baseline stays the shipped version.

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [6]:
import os
import json

os.makedirs("../outputs", exist_ok=True)
os.makedirs("../figures", exist_ok=True)

output_cols = ["content_id", "client_id", "score", "reason_code", "action",
               "impressions_90d", "ctr", "avg_position", "days_since_last_update", "trend_direction"]
ranked[output_cols].to_csv("../outputs/action_playbook_queue.csv", index=False)
print(f"Wrote {len(ranked)} rows to work/outputs/action_playbook_queue.csv")

# Metrics JSON — the receipts the paper's numbers trace back to.
metrics = {
    "eligible_rows": int(len(eligible)),
    "flagged_for_review": int((ranked["score"] > 0).sum()),
    "reason_code_counts": ranked.loc[ranked["score"]>0, "reason_code"].value_counts().to_dict(),
    "staleness_decline_rate": {
        "fresh_lt_30d": {"n": int(staleness_table.loc["fresh (<30d)","n"]), "rate": float(staleness_table.loc["fresh (<30d)","declining_rate"])},
        "moderate_30_110d": {"n": int(staleness_table.loc["moderate (30-110d)","n"]), "rate": float(staleness_table.loc["moderate (30-110d)","declining_rate"])},
        "stale_110d_plus": {"n": int(staleness_table.loc["stale (110d+)","n"]), "rate": float(staleness_table.loc["stale (110d+)","declining_rate"])},
    },
    "baseline_vs_model_note": "Per W06 audit: baseline beats W05 model on non-circular true_priority target under client-grouped split. Playbook ships the baseline.",
}

with open("../outputs/w07_playbook_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Wrote work/outputs/w07_playbook_metrics.json")
print(json.dumps(metrics, indent=2))

Wrote 30000 rows to work/outputs/action_playbook_queue.csv
Wrote work/outputs/w07_playbook_metrics.json
{
  "eligible_rows": 30000,
  "flagged_for_review": 9961,
  "reason_code_counts": {
    "low_ctr_visible_page": 6120,
    "declining_with_demand": 2094,
    "moderately_stale_page": 1747
  },
  "staleness_decline_rate": {
    "fresh_lt_30d": {
      "n": 20480,
      "rate": 0.511
    },
    "moderate_30_110d": {
      "n": 9290,
      "rate": 0.613
    },
    "stale_110d_plus": {
      "n": 230,
      "rate": 0.426
    }
  },
  "baseline_vs_model_note": "Per W06 audit: baseline beats W05 model on non-circular true_priority target under client-grouped split. Playbook ships the baseline."
}


**Note on what stays committed:** per this assignment's instructions, `action_playbook_queue.csv` stays out of git (the CI leak-guard blocks data files, and the notebook regenerates it on every run). `w07_playbook_metrics.json` and `staleness_decline_rate.png` are committed — they're the receipts and the figure next week's paper builds on.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all) — **run this yourself in Colab**; all numbers above were verified against the real starter dataset in this repo, so it should reproduce exactly
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.